# L08 · Multi-turn agent OPD

## Goal

**Estimated time:** 35 min · **Path:** full

- represent turns and environment state
- observe error compounding
- leave the single-turn assumption

### Current position: L07 → **L08** → L09

```text
Prompt/Data -> state source -> ... -> L08 -> ... -> fair evaluation
```

Alt text: The course map highlights L08 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L08"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L08', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

In multi-turn tasks, an action changes the next environment observation. One early error can create states unfamiliar to the teacher, so token loss alone cannot establish success.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

A multi-turn trajectory is not just text; it is a chain of `(observation_t, action_t, next observation, terminal)`. Student actions change transitions, so a teacher that succeeds only on the reference trajectory may not recover from student states.

Per-token turn/step boundaries reveal which step failed, whether environment tokens leaked into loss, and whether tokens after terminal were trained. Record sequence success alongside token-level teacher agreement.

### Production implementation: why this design

Environment state is immutable; `step(action)` returns the next observation plus terminal/success. Multi-turn batches give prompt tokens turn ID `-1` and response turns `0..N-1`, making mask slicing auditable.

Production code: [`calculator.py`](../../src/opd_study/envs/calculator.py), [`tokenizer.py`](../../src/opd_study/data/tokenizer.py).

In [2]:
import inspect
from opd_study.envs import CalculatorEnvironment

objects_to_show = (CalculatorEnvironment.step,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.envs.calculator.CalculatorEnvironment.step
    def step(self, proposed_value: int) -> CalculatorState:
        if self._terminal:
            raise RuntimeError("cannot step a terminal environment; call reset")
        operator, operand = self._operations[self._turn]
        correct_value = self._apply(self._value, operator, operand)
        correct = proposed_value == correct_value
        # The environment uses the student's actual value. One wrong turn therefore
        # changes every later state and makes error compounding visible.
        self._value = proposed_value
        self._turn += 1
        self._terminal = self._turn == len(self._operations)
        success = self._terminal and self._value == self._target
        if self._terminal:
            observation = (
                f"Done; value {self._value}; target {self._target}; "
                f"{'success' if success else 'failure'}"
            )
        else:
            next_operator, next_operand = self.

### Alternatives and trade-offs

A text-only transcript is compact but loses observation/action structure; full simulator snapshots are exact but large. A practical minimum preserves turn IDs, terminal state, an observation hash, and actions.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L08's output? Write one sentence, then run.

In [3]:
from opd_study.envs import CalculatorEnvironment

environment = CalculatorEnvironment(2, (("+", 3), ("*", 4)))
print(environment.reset().observation)
after_error = environment.step(6)
print(after_error.observation)
final = environment.step(24)
print(final.observation)

Current 2; apply + 3
Tool says expected 5; current 6; apply * 4
Done; value 24; target 20; failure


In [4]:
environment.reset()
correct_first = environment.step(5)
correct_final = environment.step(20)
print("correct path:", correct_first.observation, "->", correct_final.observation)
print("The next observation depends on the student's previous action.")

correct path: Tool says correct; current 5; apply * 4 -> Done; value 20; target 20; success
The next observation depends on the student's previous action.


## Checks

In [5]:
assert after_error.value == 6
assert final.target == 20 and not final.success
assert correct_final.success
print("check passed: an early action changes later states and final success")

check passed: an early action changes later states and final success


**Exercise (8 min):** construct two trajectories with the same final number but different first actions; compare observation traces.

<details><summary>Check</summary>Different environment transitions mean different provenance even if some text/final values coincide.</details>

## My recurring mistakes

### M1 — Treating multi-turn as one long text

- Wrong: discard turn and observation boundaries.
- Why: action-to-next-state causality becomes unauditable.
- Fix: preserve turn IDs, terminal state, and observations.
- Related check: `test_an_early_error_changes_later_state`

### M2 — Calling token agreement task success

- Wrong: high teacher-token agreement implies environment success.
- Why: one critical action can fail the whole task.
- Fix: report sequence success with token metrics.
- Related check: `test_an_early_error_changes_later_state`

## 60-second summary

1. represent turns and environment state
2. observe error compounding
3. leave the single-turn assumption

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`tcod`](https://arxiv.org/abs/2604.24005v3) · `2604.24005v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`sod`](https://arxiv.org/abs/2605.07725v3) · `2605.07725v3` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)
- [`sage_opd`](https://arxiv.org/abs/2606.19659v1) · `2606.19659v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)